In [14]:
!pip install -q streamlit
!pip install -q google-genai
!pip install -q pyngrok
!pip install -q nest_asyncio

In [15]:
import subprocess
import threading
import time
import os
import nest_asyncio

from pyngrok import ngrok

In [16]:
nest_asyncio.apply()

In [17]:
NGROK_AUTH_TOKEN = "API NGOROK"

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

print("✅ Ngrok Connected")

✅ Ngrok Connected


In [18]:
import subprocess
import threading
import time
import os

def run_streamlit(filename):

    # Menjalankan Streamlit
    proc = subprocess.Popen(
        [
            "streamlit",
            "run",
            filename,
            "--server.port",
            "8501",
            "--server.address",
            "0.0.0.0"
        ]
    )

    # Tunggu server aktif
    time.sleep(5)

    # Membuka tunnel ngrok
    public_url = ngrok.connect(8501)

    print("="*60)
    print("🎓 EduAI Berhasil Dijalankan")
    print("="*60)
    print(f"🌐 URL : {public_url}")
    print("="*60)

    return proc, public_url

In [19]:
%%writefile streamlit_chat_app.py

# ==========================================================
# EduAI - Education Chatbot
# Dibuat menggunakan:
# Streamlit + Google Gemini + Google Colab
# ==========================================================

import streamlit as st
from google import genai

# ==========================================================
# Konfigurasi Halaman
# ==========================================================

st.set_page_config(
    page_title="EduAI",
    page_icon="🎓",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ==========================================================
# Custom CSS
# ==========================================================

st.markdown("""
<style>

.stApp{
    background:#09090B;
    color:white;
}

.block-container{
    padding-top:1.5rem;
}

header{
    visibility:hidden;
}

footer{
    visibility:hidden;
}

#MainMenu{
    visibility:hidden;
}

/* Sidebar */

section[data-testid="stSidebar"]{
    background:#111827;
    border-right:1px solid #2563EB;
}

/* Input */

.stTextInput input{
    background:#18181B;
    color:white;
    border-radius:10px;
}

/* Button */

.stButton>button{
    width:100%;
    background:#2563EB;
    color:white;
    border-radius:10px;
    border:none;
    font-weight:bold;
}

.stButton>button:hover{
    background:#1D4ED8;
}

/* Chat */

.stChatMessage{
    border-radius:15px;
}

/* Card */

.card{

    background:#111827;

    border:1px solid #2563EB;

    padding:18px;

    border-radius:15px;

}

.bigtitle{

    text-align:center;

    color:white;

    font-size:42px;

    font-weight:800;

}

.subtitle{

    text-align:center;

    color:#9CA3AF;

    font-size:17px;

    margin-bottom:20px;

}

</style>
""", unsafe_allow_html=True)

# ==========================================================
# Session State
# ==========================================================

if "genai_client" not in st.session_state:
    st.session_state.genai_client = None

if "_last_key" not in st.session_state:
    st.session_state._last_key = ""

if "chat" not in st.session_state:
    st.session_state.chat = None

if "messages" not in st.session_state:
    st.session_state.messages = []

if "memory" not in st.session_state:
    st.session_state.memory = {
        "student_level":"",
        "favorite_subject":"",
        "name":""
    }

# ==========================================================
# Header
# ==========================================================

st.markdown("""

<div class="bigtitle">

🎓 EduAI

</div>

<div class="subtitle">

Teman Belajar Pintar untuk Pelajar Indonesia

</div>

""", unsafe_allow_html=True)

# ==========================================================
# Sidebar
# ==========================================================

with st.sidebar:

    st.title("🎓 EduAI")

    st.caption("Education Chatbot")

    st.divider()

    st.subheader("🔑 Google API")

    google_api_key = st.text_input(

        "Masukkan Google Gemini API",

        type="password",

        placeholder="AIzaSy..."

    )

    connect = st.button("Hubungkan EduAI")

    st.divider()

    st.subheader("📊 Status")

    if st.session_state.genai_client is None:

        st.error("Belum Terhubung")

    else:

        st.success("Sudah Terhubung")

    st.write("Model")

    st.info("Gemini 2.5 Flash")

    st.write("Memory")

    st.info("Aktif")

    st.write("Jumlah Chat")

    st.info(len(st.session_state.messages))

    st.divider()

    reset_chat = st.button("🧹 Reset Chat")

    reset_memory = st.button("🗑 Reset Memory")

    st.divider()

    st.caption("EduAI v1.0")

    # ==========================================================
# Validasi API
# ==========================================================

if connect:

    if google_api_key == "":

        st.warning("Silakan masukkan Google API Key terlebih dahulu.")

    else:

        try:

            st.session_state.genai_client = genai.Client(
                api_key=google_api_key
            )

            st.session_state._last_key = google_api_key

            st.success("🎉 EduAI berhasil terhubung!")

        except Exception as e:

            st.error(f"Gagal terhubung.\n\n{e}")

# ==========================================================
# Belum connect
# ==========================================================

if st.session_state.genai_client is None:

    st.markdown("""

<div class="card">

<h3>👋 Selamat Datang di EduAI</h3>

Masukkan <b>Google Gemini API Key</b> melalui sidebar untuk mulai belajar bersama EduAI.

Fitur:

<ul>

<li>📚 Membantu memahami materi sekolah</li>

<li>📝 Menjelaskan langkah demi langkah</li>

<li>🧠 Memory selama sesi chat</li>

<li>💡 Bahasa santai dan mudah dipahami</li>

</ul>

</div>

""", unsafe_allow_html=True)

    st.stop()

# ==========================================================
# RESET CHAT & MEMORY
# ==========================================================

if reset_chat:

    st.session_state.messages = []

    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-2.5-flash"
    )

    st.rerun()


if reset_memory:

    st.session_state.messages = []

    st.session_state.memory = {
        "student_level": "",
        "favorite_subject": "",
        "name": ""
    }

    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-2.5-flash"
    )

    st.success("Memory berhasil dihapus.")

    st.rerun()


# ==========================================================
# MEMBUAT CHAT GEMINI
# ==========================================================

if st.session_state.chat is None:

    st.session_state.chat = st.session_state.genai_client.chats.create(
        model="gemini-2.5-flash"
    )


# ==========================================================
# TAMPILKAN HISTORY CHAT
# ==========================================================

for message in st.session_state.messages:

    with st.chat_message(message["role"]):

        st.markdown(message["content"])


# ==========================================================
# INPUT USER
# ==========================================================

prompt = st.chat_input(
    "Tanyakan materi sekolah..."
)


if prompt:

    # ----------------------------
    # Simpan chat user
    # ----------------------------

    st.session_state.messages.append(
        {
            "role":"user",
            "content":prompt
        }
    )

    with st.chat_message("user"):

        st.markdown(prompt)


    # ----------------------------
    # Memory sederhana
    # ----------------------------

    text = prompt.lower()

    if "nama saya" in text:

        try:

            nama = prompt.split("nama saya")[-1]

            st.session_state.memory["name"] = nama.strip()

        except:

            pass


    if "kelas 10" in text:

        st.session_state.memory["student_level"] = "Kelas 10"

    elif "kelas 11" in text:

        st.session_state.memory["student_level"] = "Kelas 11"

    elif "kelas 12" in text:

        st.session_state.memory["student_level"] = "Kelas 12"


    daftar_mapel = [

        "matematika",
        "fisika",
        "kimia",
        "biologi",
        "informatika",
        "bahasa inggris",
        "sejarah",
        "ekonomi"

    ]

    for mapel in daftar_mapel:

        if mapel in text:

            st.session_state.memory["favorite_subject"] = mapel


    # ----------------------------
    # Prompt EduAI
    # ----------------------------

    system_prompt = f"""
Kamu adalah EduAI.

EduAI adalah chatbot pendidikan khusus pelajar Indonesia.

Tugasmu adalah membantu siswa memahami materi pelajaran.

Gunakan Bahasa Indonesia.

Gunakan bahasa santai.

Gunakan sedikit bahasa gaul.

Tetap sopan.

Berikan penjelasan langkah demi langkah.

Jika menjelaskan matematika,
berikan proses pengerjaannya.

Jika user meminta jawaban PR,
bantu memahami konsepnya terlebih dahulu.

Gunakan emoji seperlunya.

==========================

Memory User

Nama:
{st.session_state.memory['name']}

Jenjang:
{st.session_state.memory['student_level']}

Pelajaran Favorit:
{st.session_state.memory['favorite_subject']}

==========================

Selalu gunakan memory tersebut apabila relevan.
"""


    final_prompt = f"""

{system_prompt}

Pertanyaan User:

{prompt}

"""


    # ----------------------------
    # Jawaban AI
    # ----------------------------

    with st.chat_message("assistant"):

        loading = st.empty()

        loading.info("EduAI sedang berpikir...")

        try:

            response = st.session_state.chat.send_message(
                final_prompt
            )

            if hasattr(response, "text"):

                answer = response.text

            else:

                answer = str(response)

        except Exception as e:

            answer = f"Terjadi error:\n\n{e}"

        loading.empty()

        st.markdown(answer)


    # ----------------------------
    # Simpan history
    # ----------------------------

    st.session_state.messages.append(
        {
            "role":"assistant",
            "content":answer
        }
    )

# ==========================================================
# HOME CARD
# ==========================================================

if len(st.session_state.messages) == 0:

    st.markdown("## 🚀 Mulai Belajar Bersama EduAI")

    col1, col2 = st.columns(2)

    with col1:

        st.markdown("""
<div class="card">

### 📖 Ringkas Materi

Contoh:

- Ringkas Sistem Pernapasan
- Ringkas Revolusi Industri
- Ringkas Integral

</div>
""", unsafe_allow_html=True)

        st.markdown("""
<div class="card">

### 📚 Penjelasan Mudah

EduAI akan menjelaskan seperti guru yang sabar menggunakan bahasa santai.

</div>
""", unsafe_allow_html=True)

    with col2:

        st.markdown("""
<div class="card">

### 📝 Latihan Soal

Contoh:

- Buat 10 soal Matematika
- Buat latihan Informatika
- Soal HOTS Biologi

</div>
""", unsafe_allow_html=True)

        st.markdown("""
<div class="card">

### 💡 Contoh Prompt

• Jelaskan Fotosintesis

• Apa itu Algoritma?

• Ringkas Perang Dunia II

• Ajari aku Python dari nol

• Buat latihan kelas 11

</div>
""", unsafe_allow_html=True)

# ==========================================================
# MEMORY VIEW
# ==========================================================

st.divider()

with st.expander("🧠 Memory EduAI"):

    st.write("Nama")

    st.code(
        st.session_state.memory["name"]
        if st.session_state.memory["name"]
        else "-"
    )

    st.write("Jenjang")

    st.code(
        st.session_state.memory["student_level"]
        if st.session_state.memory["student_level"]
        else "-"
    )

    st.write("Pelajaran Favorit")

    st.code(
        st.session_state.memory["favorite_subject"]
        if st.session_state.memory["favorite_subject"]
        else "-"
    )

# ==========================================================
# FOOTER
# ==========================================================

st.divider()

st.markdown(
"""
<div style="text-align:center;color:gray;">

🎓 <b>EduAI</b><br>

Education Chatbot berbasis Google Gemini.<br>

Dibangun menggunakan Streamlit + Google AI Studio.

</div>
""",
unsafe_allow_html=True
)




Overwriting streamlit_chat_app.py


In [20]:
proc, url = run_streamlit("streamlit_chat_app.py")

🎓 EduAI Berhasil Dijalankan
🌐 URL : NgrokTunnel: "https://deflected-dazzler-villain.ngrok-free.dev" -> "http://localhost:8501"


In [21]:
print("==========================================")
print("🎓 EduAI Berhasil Berjalan")
print("==========================================")
print(url)
print("==========================================")

🎓 EduAI Berhasil Berjalan
NgrokTunnel: "https://deflected-dazzler-villain.ngrok-free.dev" -> "http://localhost:8501"
